# Notebook: Run All


In [ ]:
import glob
import os
import subprocess
import sys

import nbformat
from nbclient import NotebookClient


sys.path.append(os.path.abspath(".."))
sys.path.append(os.path.abspath("../src"))
from src.utils.helpers import p, t, c


t("Run All Notebooks")

root = os.path.dirname(os.getcwd())
source_folder = os.path.join(root, "notebooks")
p("source_folder", source_folder)
p()
notebooks = glob.glob(os.path.join(source_folder, "*.ipynb"))
#notebooks = [nb for nb in notebooks if os.path.basename(nb) != "_run_all.ipynb"]
notebooks = [nb for nb in notebooks if os.path.basename(nb) != "_run_all.ipynb"
             #and all(s not in os.path.basename(nb) for s in ("_32", "_512"))
             ]
notebooks = sorted(notebooks)

for nb in notebooks:
    print(os.path.basename(nb))



In [ ]:
failed_notebooks = []

for nb_path in notebooks:
    nb_name = os.path.basename(nb_path)
    try:
        p()
        t(f"Executing: {nb_name}")

        nb = nbformat.read(nb_path, as_version = 4)
        client = NotebookClient(nb, timeout = 900, kernel_name = "python3")
        client.execute()

        # Save executed notebook
        nbformat.write(nb, nb_path)
        p(f"Finished & saved: {nb_name}", color1 = c.BLUE)

        # Path relative to repo root for git
        rel_nb_path = os.path.relpath(nb_path, root)

        # Add, commit, push
        subprocess.run(["git", "add", rel_nb_path], check = True)
        commit_msg = f"Auto-update {nb_name}"
        subprocess.run(["git", "commit", "-m", commit_msg], check = False)
        subprocess.run(["git", "push"], check = True)

        p(f"Committed and pushed: {nb_name}", color1 = c.GREEN)

    except Exception as e:
        p(f"Error while executing {nb_name}", str(e), color1 = c.RED, color2 = c.BLACK)
        failed_notebooks.append(nb_path)

t("\nExecution completed.")

if failed_notebooks:
    p("\n\nNotebooks that failed:", color1 = c.MAGENTA)
    for nb_path in failed_notebooks:
        p("", nb_path)
